<a href="https://colab.research.google.com/github/Miranics/TradesREvamp/blob/main/research/03_pyramiding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pyramiding: how hard to press winners

Same sweep model and same Exness data as the bake-off, but seven ways of adding to winning trades, from no adds to adding every 0.5R and holding overnight. Uses the gold data already in `My Trades/data`, so there's nothing to download (run `Strategy_Bakeoff` first if that folder is empty).

Run all cells, then paste everything printed from Step 5 onwards back to the chat.

**Step 1.** Connect Google Drive (approve the pop-up).

In [ ]:
import glob
import os

from google.colab import drive

drive.mount("/content/drive")

matches = [p for p in glob.glob("/content/drive/MyDrive/*") if os.path.basename(p).strip() == "My Trades"]
if not matches:
    raise FileNotFoundError("Could not find a folder called 'My Trades' at the top of My Drive")
BASE = matches[0]
DATA = os.path.join(BASE, "data")
OUT = os.path.join(BASE, "results")
os.makedirs(DATA, exist_ok=True)
os.makedirs(OUT, exist_ok=True)
print("Data folder:   ", DATA)
print("Results folder:", OUT)


**Step 2.** Load the Exness gold data saved by the bake-off.

In [ ]:
import glob
import os

import numpy as np
import pandas as pd

SPREAD = 0.30
SPREAD_HIGH = 0.60
OZ_PER_001_LOT = 1


def load_m1(pattern):
    files = sorted(glob.glob(os.path.join(DATA, pattern)))
    if not files:
        raise FileNotFoundError(f"No files matching {pattern} in {DATA}")
    df = pd.concat(pd.read_csv(f) for f in files)
    df["time"] = pd.to_datetime(df["timestamp"], unit="ms", utc=True)
    df = df.drop_duplicates("time").set_index("time").sort_index()
    keep = [c for c in ("open", "high", "low", "close", "spread") if c in df.columns]
    df = df[keep].astype(float)
    return df[df.index.dayofweek < 5]


m1 = load_m1("xauusd_*.csv")
m1_ny = m1.tz_convert("America/New_York")

if "spread" in m1.columns:
    ny_spread = m1.loc[(m1.index.hour >= 13) & (m1.index.hour < 21), "spread"].median()
    SPREAD = round(float(ny_spread), 2)
    SPREAD_HIGH = round(2 * SPREAD, 2)
print(f"Cost per position: ${SPREAD:.2f} (Exness median spread in New York hours); stress test: ${SPREAD_HIGH:.2f}")

per_month = m1.groupby(m1.index.strftime("%Y-%m")).size()
print(f"Gold minutes loaded: {len(m1):,}  from {m1.index[0]:%Y-%m-%d} to {m1.index[-1]:%Y-%m-%d}")
thin = per_month.iloc[:-1][per_month.iloc[:-1] < 15000]
if len(thin):
    print("Months with too little data (re-run Step 2 to fill them):", ", ".join(thin.index))
else:
    print("Every month looks complete.")


**Step 3.** Load the trading logic (prints nothing). Identical to the bake-off.

In [ ]:
DISPLACEMENT = 1.5
BUFFER = 0.50


def bracket(d, entry, stop, path, rr=None):
    """Gross points for one position entered at `entry` on the first bar of `path`.
    Stop and target are checked on every bar; if both are touched in the same bar the stop wins."""
    risk = abs(entry - stop)
    if risk <= 0 or len(path) == 0:
        return None
    hi, lo = path["high"].to_numpy(), path["low"].to_numpy()
    n = len(path)
    sl = (lo <= stop) if d == 1 else (hi >= stop)
    i_sl = int(np.argmax(sl)) if sl.any() else n
    i_tp = n
    if rr is not None:
        target = entry + d * rr * risk
        tp = (hi >= target) if d == 1 else (lo <= target)
        i_tp = int(np.argmax(tp)) if tp.any() else n
    if i_tp < i_sl:
        return rr * risk
    if i_sl < n:
        return -risk
    return d * (path["close"].iloc[-1] - entry)


def swing_points(b, side):
    s = b["low"] if side == "low" else b["high"]
    if side == "low":
        return (s < s.shift(1)) & (s < s.shift(2)) & (s < s.shift(-1)) & (s < s.shift(-2))
    return (s > s.shift(1)) & (s > s.shift(2)) & (s > s.shift(-1)) & (s > s.shift(-2))


def find_setup(b, lh, ll):
    body = (b["close"] - b["open"]).abs()
    avg_body = body.rolling(20, min_periods=5).median().shift(1)
    ny = b[(b.index.hour >= 13) & (b.index.hour < 17)]
    for t in ny.index:
        i = b.index.get_loc(t)
        bar = b.iloc[i]
        for d, swept in ((-1, bar["high"] > lh), (1, bar["low"] < ll)):
            if not swept:
                continue
            prior = b.iloc[max(0, i - 40):i]
            sw = swing_points(b.iloc[max(0, i - 42):i + 1], "low" if d == -1 else "high")
            sw = sw.reindex(prior.index).fillna(False).astype(bool)
            if not sw.any():
                continue
            level = prior.loc[sw[sw].index[-1], "low" if d == -1 else "high"]
            extreme = bar["high"] if d == -1 else bar["low"]
            for j in range(i + 1, min(i + 30, len(b))):
                x = b.iloc[j]
                extreme = max(extreme, x["high"]) if d == -1 else min(extreme, x["low"])
                broke = x["close"] < level if d == -1 else x["close"] > level
                ab = avg_body.iloc[j]
                strong = body.iloc[j] >= DISPLACEMENT * (ab if ab and not np.isnan(ab) else np.inf)
                if broke and strong:
                    for k in range(j + 1, i, -1):
                        if k >= len(b) or k - 2 < 0:
                            continue
                        a, c = b.iloc[k - 2], b.iloc[k]
                        if d == -1 and a["low"] > c["high"]:
                            return dict(dir=d, entry=c["high"], stop=extreme + BUFFER,
                                        ready=b.index[min(k + 1, len(b) - 1)])
                        if d == 1 and a["high"] < c["low"]:
                            return dict(dir=d, entry=c["low"], stop=extreme - BUFFER,
                                        ready=b.index[min(k + 1, len(b) - 1)])
                    return None
            return None
    return None


def find_fill(s, d1):
    d, entry, stop = s["dir"], s["entry"], s["stop"]
    after = d1[(d1.index >= s["ready"]) & (d1.index.hour < 21)]
    for t, x in after.iterrows():
        if (d == -1 and x["high"] >= stop) or (d == 1 and x["low"] <= stop):
            return None
        if (d == -1 and x["high"] >= entry) or (d == 1 and x["low"] <= entry):
            return None if t.hour >= 20 else t
    return None


def run_from_fill(d, entry, stop, d1, fill_t, adds=()):
    """Gross points and number of positions for a trade filled at fill_t, managed to 21:00 UTC."""
    risk = abs(entry - stop)
    window = d1[(d1.index >= fill_t) & (d1.index.hour < 21)]
    after = window[window.index > fill_t]
    units, todo = [entry], list(adds)
    for t, x in after.iterrows():
        hit_sl = x["high"] >= stop if d == -1 else x["low"] <= stop
        if hit_sl:
            return sum(d * (stop - e) for e in units), len(units)
        best = (entry - x["low"]) if d == -1 else (x["high"] - entry)
        while todo and best >= todo[0] * risk:
            units.append(entry + d * todo.pop(0) * risk)
            stop = entry
    last = window["close"].iloc[-1]
    return sum(d * (last - e) for e in units), len(units)


**Step 4.** Find every sweep-model setup from 2024 to 2026, once (a few minutes).

In [ ]:
setups = []
days = sorted(set(m1.index.date))
for i, (day, d1) in enumerate(m1.groupby(m1.index.date)):
    london = d1[(d1.index.hour >= 7) & (d1.index.hour < 12)]
    if len(london) < 200:
        continue
    b = d1.resample("5min").agg({"open": "first", "high": "max", "low": "min", "close": "last"}).dropna()
    s = find_setup(b, london["high"].max(), london["low"].min())
    if not s:
        continue
    fill_t = find_fill(s, d1)
    if fill_t is None:
        continue
    j = days.index(day)
    next_day = days[j + 1] if j + 1 < len(days) else day
    end_next = pd.Timestamp(next_day).tz_localize("UTC") + pd.Timedelta(hours=21)
    end_same = pd.Timestamp(day).tz_localize("UTC") + pd.Timedelta(hours=21)
    path = m1[(m1.index >= fill_t) & (m1.index < end_next)]
    setups.append(dict(day=pd.Timestamp(day), year=day.year, dir=s["dir"], entry=s["entry"], stop=s["stop"],
                       risk=abs(s["entry"] - s["stop"]), fill_t=fill_t, end_same=end_same, path=path))

print(f"Setups found and filled: {len(setups)}")
print(pd.Series([s["year"] for s in setups]).value_counts().sort_index().to_string())


**Step 5.** Compare the seven ways of managing the same trades.

In [ ]:
def manage(s, adds=(), sizes=(1,), trail=False, overnight=False):
    """Net result in R (1R = first position's risk) and total size, for one setup."""
    d, entry, stop, risk = s["dir"], s["entry"], s["stop"], s["risk"]
    path = s["path"] if overnight else s["path"][s["path"].index < s["end_same"]]
    after = path[path.index > s["fill_t"]]
    hi, lo = after["high"].to_numpy(), after["low"].to_numpy()
    units, todo = [(entry, sizes[0])], list(adds)
    exit_price = path["close"].iloc[-1]
    for k in range(len(after)):
        if (d == 1 and lo[k] <= stop) or (d == -1 and hi[k] >= stop):
            exit_price = stop
            break
        best = (hi[k] - entry) if d == 1 else (entry - lo[k])
        while todo and best >= todo[0] * risk:
            level = entry + d * todo.pop(0) * risk
            units.append((level, sizes[min(len(units), len(sizes) - 1)]))
            stop = units[-2][0] if trail else entry
    gross = sum(sz * d * (exit_price - e) for e, sz in units)
    size = sum(sz for _, sz in units)
    return (gross - size * SPREAD) / risk, size


VARIANTS = {
    "1 No adds, hold to 23:00":                 dict(),
    "2 Adds at +1R, +2R (current plan)":        dict(adds=(1, 2), sizes=(1, 1, 1)),
    "3 Adds every 1R up to +4R":                dict(adds=(1, 2, 3, 4), sizes=(1,)),
    "4 Adds every 0.5R up to +3R":              dict(adds=(0.5, 1, 1.5, 2, 2.5, 3), sizes=(1,)),
    "5 Adds every 1R up to +4R, trailing stop": dict(adds=(1, 2, 3, 4), sizes=(1,), trail=True),
    "6 Growing adds: sizes 1, 2, 3":            dict(adds=(1, 2), sizes=(1, 2, 3)),
    "7 Adds up to +4R, hold to next day 23:00": dict(adds=(1, 2, 3, 4), sizes=(1,), overnight=True),
}

res = []
for name, kw in VARIANTS.items():
    for s in setups:
        r, size = manage(s, **kw)
        res.append(dict(variant=name, day=s["day"], year=s["year"], R=r, size=size, risk=s["risk"]))
res = pd.DataFrame(res)


def summary(g):
    g = g.sort_values("day")
    eq = g["R"].cumsum()
    losing = (g["R"] <= 0).astype(int)
    streak = losing.groupby((losing == 0).cumsum()).sum().max()
    top5 = g["R"].nlargest(5).sum() / g["R"].sum() * 100 if g["R"].sum() > 0 else np.nan
    return pd.Series({"trades": len(g), "win_%": round((g["R"] > 0).mean() * 100, 1),
                      "avg_R": round(g["R"].mean(), 3), "total_R": round(g["R"].sum(), 1),
                      "best_R": round(g["R"].max(), 1), "max_drawdown_R": round((eq.cummax() - eq).max(), 1),
                      "longest_losing_run": int(streak), "top5_share_%": round(top5, 0)})


pd.set_option("display.width", 220)
print(f"Cost per position: ${SPREAD:.2f}. 1R = distance from first entry to the original stop.\n")
print("## Average R per trade by year\n")
print(res.pivot_table(index="variant", columns="year", values="R", aggfunc="mean").round(3).to_string())
print("\n## All years combined\n")
print(res.groupby("variant").apply(summary, include_groups=False).to_string())
print("\ntop5_share_% = share of all profit that came from the 5 best trades.")


**Step 6.** What each version does to a $50 attempt with your sizing plan (0.01 lot below $250, then 20% risk).

In [ ]:
START_BALANCE = 50
SEED_UNTIL = 250
GROWTH_RISK = 0.20
TARGETS = (250, 1000)
MAX_TRADES = 150
N_SIMS = 4000
rng = np.random.default_rng(7)


def simulate_account(R, risk_usd):
    """One $50 attempt: 0.01 lot per position below $250, then 20% of balance at risk per position."""
    bal, hit = START_BALANCE, {t: False for t in TARGETS}
    for _ in range(MAX_TRADES):
        i = rng.integers(len(R))
        oz = 1 if bal < SEED_UNTIL else max(1, int(bal * GROWTH_RISK / risk_usd[i]))
        if risk_usd[i] * oz > bal:
            if bal < risk_usd.min():
                return hit, True
            continue
        bal += R[i] * risk_usd[i] * oz
        for t in TARGETS:
            hit[t] = hit[t] or bal >= t
        if bal < 5:
            return hit, True
        if hit[TARGETS[-1]]:
            break
    return hit, False


recent = res[res["year"] >= 2025]
rows = []
for name, g in recent.groupby("variant"):
    R, risk_usd = g["R"].to_numpy(), g["risk"].to_numpy()
    out = [simulate_account(R, risk_usd) for _ in range(N_SIMS)]
    p250 = np.mean([h[250] for h, _ in out])
    rows.append({"variant": name,
                 "reach_$250_%": round(100 * p250, 1),
                 "reach_$1000_%": round(100 * np.mean([h[1000] for h, _ in out]), 1),
                 "bust_%": round(100 * np.mean([b for _, b in out]), 1),
                 "monthly_$250_with_3_attempts_%": round(100 * (1 - (1 - p250) ** 3), 1)})

print(f"## A $50 attempt under your sizing plan ({N_SIMS:,} simulated attempts per variant)\n")
print("Trades are drawn at random from 2025-2026 results, with their real stop sizes.\n")
print(pd.DataFrame(rows).to_string(index=False))
print("\nThis assumes future trades look like 2025-2026. A different market will give different odds.")

res.to_csv(os.path.join(OUT, "pyramiding_trades.csv"), index=False)
print(f"\nSaved pyramiding_trades.csv to {OUT}")
